# Comparação dos resultados de validação

Este notebook apenas lê artefatos já salvos dos quatro experimentos. Ele não treina modelos e não calcula métricas no conjunto de teste. No Colab, usa por padrão o mesmo projeto persistente em `MyDrive/plant-disease-classification` dos notebooks de treinamento.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPOSITORY_URL = 'https://github.com/murilodc/plant-disease-classification.git'
COLAB_PROJECT_DIR = Path('/content/plant-disease-classification')
LEGACY_COLAB_PROJECT_DIR = Path('/content/tcc-plant-disease-classification')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/plant-disease-classification')
LEGACY_DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/TCC')
AUTO_CLONE_REPOSITORY = True
MOUNT_DRIVE = True
if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')


def is_project_root(path: Path) -> bool:
    return (path / 'src' / 'compare_models.py').is_file()


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    if IN_COLAB:
        candidates.extend([
            DRIVE_PROJECT_DIR, LEGACY_DRIVE_PROJECT_DIR,
            COLAB_PROJECT_DIR, LEGACY_COLAB_PROJECT_DIR,
        ])
    for candidate in candidates:
        if is_project_root(candidate):
            return candidate

    if IN_COLAB and AUTO_CLONE_REPOSITORY:
        clone_target = DRIVE_PROJECT_DIR if MOUNT_DRIVE else COLAB_PROJECT_DIR
        if clone_target.exists():
            raise FileNotFoundError(
                f'{clone_target} já existe, mas não contém o projeto completo. '
                'Remova ou renomeie esse diretório e execute a célula novamente.'
            )
        print('Clonando o repositório para o Colab...')
        subprocess.check_call([
            'git', 'clone', '--depth', '1', REPOSITORY_URL, str(clone_target)
        ])
        if is_project_root(clone_target):
            return clone_target

    raise FileNotFoundError(
        'Projeto não encontrado. Envie o repositório completo, monte o Drive '
        'ou habilite AUTO_CLONE_REPOSITORY.'
    )


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / 'results'
COMPARISON_DIR = RESULTS_DIR / 'model_comparison'
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

required_packages = {
    'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib'
}
missing_packages = [
    package for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

from IPython.display import Image, display

print(f'Projeto: {PROJECT_ROOT}')
print(f'Resultados: {RESULTS_DIR}')

In [ ]:
from compare_models import (
    compare_validation_results,
    save_comparison_plots,
    save_training_curve_plots,
)

## Carregamento e situação dos experimentos

In [ ]:
comparison = compare_validation_results(
    results_root=RESULTS_DIR,
    output_dir=COMPARISON_DIR,
    save=True,
)
display(comparison.summary[['model_name', 'status', 'missing_artifacts']])

In [ ]:
if comparison.warnings:
    print('Avisos de integridade/comparabilidade:')
    for warning in comparison.warnings:
        print(f'- {warning}')
else:
    print('Nenhum aviso de integridade/comparabilidade.')

## Tabela resumo

In [ ]:
display(comparison.summary)
print(f'Resumo salvo em: {comparison.summary_path}')

## Comparações objetivas

In [ ]:
comparison_plot_paths = save_comparison_plots(
    comparison.summary,
    COMPARISON_DIR,
)
if not comparison_plot_paths:
    print('Ainda não há métricas completas para gerar os gráficos.')
for plot_path in comparison_plot_paths:
    display(Image(filename=str(plot_path)))

## Curvas nas épocas realmente executadas

In [ ]:
curve_plot_paths = save_training_curve_plots(
    comparison.histories,
    COMPARISON_DIR,
)
if not curve_plot_paths:
    print('Ainda não há históricos válidos para gerar as curvas.')
for plot_path in curve_plot_paths:
    display(Image(filename=str(plot_path)))

## Métricas de validação por classe

In [ ]:
if comparison.per_class.empty:
    print('Ainda não há métricas por classe disponíveis.')
else:
    display(comparison.per_class)
print(f'Tabela por classe salva em: {comparison.per_class_path}')

## Artefatos produzidos

In [ ]:
generated_paths = [
    comparison.summary_path,
    comparison.per_class_path,
    comparison.warnings_path,
    *comparison_plot_paths,
    *curve_plot_paths,
]
for path in generated_paths:
    if path is not None:
        print(path)